<a href="https://colab.research.google.com/github/Peanut2Butter9/thesis/blob/main/MacMahon_Partitions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MacMahon's partition functions and prime detection

Companion notebook to Sections 3.5 and 4.1.1 of the bachelor's thesis
*Prime-Detecting Quasimodular Forms and Some of Their Congruence Properties*
(Moh Hasbi Alfarizi, Universitas Gadjah Mada, 2026).

It computes MacMahon's partition functions $M_a(n)$, checks the polynomial identities of
Theorem 4.1.14 that vanish exactly at the primes, and checks a few congruences for $M_2$ and $M_3$.

**Requirements:** Python 3 only, no packages. Runs in a few seconds, also on Google Colab.

## 1. MacMahon's partition functions

For $a \ge 1$, MacMahon's partition function is

$$M_a(n) = \sum_{\substack{0 < s_1 < s_2 < \cdots < s_a \\ n = m_1 s_1 + m_2 s_2 + \cdots + m_a s_a}} m_1 m_2 \cdots m_a .$$

The sum runs over the partitions of $n$ with exactly $a$ distinct part sizes $s_1 < \dots < s_a$,
where $s_i$ appears $m_i \ge 1$ times, and each partition is weighted by the product of its
multiplicities. For example, $M_1(n) = \sigma_1(n)$, the sum of the divisors of $n$.

`MacMahon(a, n)` computes $M_a(n)$ by enumerating these partitions directly.

In [1]:
def MacMahon(a, n):
    # M_a(n) by direct enumeration of the partitions of n with exactly a distinct part sizes
    if a <= 0 or n <= 0:
        raise ValueError("a and n must be positive integers")
    total_sum = [0]
    _find_partitions_recursive(n, a, 1, [], total_sum)
    return total_sum[0]

def _find_partitions_recursive(target_n, blocks_left, min_s, heights, total_sum):
    # heights holds the multiplicities chosen so far; the next part size is at least min_s
    if blocks_left == 1:
        for s in range(min_s, target_n + 1):
            if target_n % s == 0:
                product = target_n // s
                for h in heights:
                    product *= h
                total_sum[0] += product
        return

    for s in range(min_s, target_n):
        # the remaining blocks_left - 1 part sizes are distinct and larger than s
        min_possible_sum = sum(range(s + 1, s + blocks_left))
        for d in range(1, (target_n // s) + 1):
            remaining_n = target_n - d * s
            if remaining_n >= min_possible_sum:
                _find_partitions_recursive(remaining_n, blocks_left - 1, s + 1, heights + [d], total_sum)

print([MacMahon(1, n) for n in range(1, 13)])
print([MacMahon(2, n) for n in range(1, 13)])
print([MacMahon(3, n) for n in range(1, 13)])

[1, 3, 4, 7, 6, 12, 8, 15, 13, 18, 12, 28]
[0, 0, 1, 3, 9, 15, 30, 45, 67, 99, 135, 175]
[0, 0, 0, 0, 0, 1, 3, 9, 22, 42, 81, 140]


Direct enumeration gets slow for large $n$. The generating function

$$U_a(q) = \sum_{n \ge 1} M_a(n)\, q^n = \sum_{0 < s_1 < \cdots < s_a} \prod_{i=1}^{a} \frac{q^{s_i}}{(1 - q^{s_i})^2},
\qquad \frac{q^{s}}{(1 - q^{s})^2} = \sum_{m \ge 1} m\, q^{ms},$$

gives a faster method: `macmahon_table(a_max, N)` expands $\prod_{s \ge 1} \left(1 + x\,\frac{q^s}{(1-q^s)^2}\right)$,
whose coefficient of $x^a q^n$ is $M_a(n)$, and returns all values with $a \le a_{\max}$ and $n \le N$ at once.

In [2]:
def macmahon_table(a_max, N):
    # M[a][n] = M_a(n) for 0 <= a <= a_max and 0 <= n <= N, with M_0(0) = 1
    M = [[0] * (N + 1) for _ in range(a_max + 1)]
    M[0][0] = 1
    for s in range(1, N + 1):
        # go down in a so that M[a - 1] does not yet use the part size s
        for a in range(a_max, 0, -1):
            row, prev = M[a], M[a - 1]
            for n in range(s, N + 1):
                row[n] += sum(m * prev[n - m*s] for m in range(1, n // s + 1))
    return M

def sigma1(n):
    return sum(d for d in range(1, n + 1) if n % d == 0)

def is_prime(n):
    return n > 1 and all(n % p for p in range(2, int(n**0.5) + 1))

N = 1000
M = macmahon_table(4, N)

assert all(M[a][n] == MacMahon(a, n) for a in range(1, 4) for n in range(1, 61))
assert all(M[1][n] == sigma1(n) for n in range(1, N + 1))
print(f"macmahon_table agrees with MacMahon for a <= 3, n <= 60, and M_1(n) = sigma_1(n) for n <= {N}.")

macmahon_table agrees with MacMahon for a <= 3, n <= 60, and M_1(n) = sigma_1(n) for n <= 1000.


## 2. Polynomials in $M_a(n)$ that detect the primes

Theorem 4.1.14 (from Craig, van Ittersum and Ono, 2024) states that for $n \ge 2$

$$(n^2 - 3n + 2)\,M_1(n) - 8\,M_2(n) = 0$$

and

$$(3n^3 - 13n^2 + 18n - 8)\,M_1(n) + (12n^2 - 120n + 212)\,M_2(n) - 960\,M_3(n) = 0$$

each hold if and only if $n$ is prime. The cell below checks both, and a third identity of the same
kind involving $M_4$, for $2 \le n \le 1000$.

In [3]:
def f(n):
    return (n**2 - 3*n + 2)*M[1][n] - 8*M[2][n]

def g(n):
    return (3*n**3 - 13*n**2 + 18*n - 8)*M[1][n] + (12*n**2 - 120*n + 212)*M[2][n] - 960*M[3][n]

def h(n):
    return ((25*n**4 - 171*n**3 + 423*n**2 - 447*n + 170)*M[1][n]
            + (300*n**3 - 3554*n**2 + 12900*n - 14990)*M[2][n]
            + (2400*n**2 - 60480*n + 214080)*M[3][n]
            - 725760*M[4][n])

print(f"{'n':>3} {'prime':>6} {'f(n)':>8} {'g(n)':>10} {'h(n)':>13}")
for n in range(2, 21):
    print(f"{n:>3} {'yes' if is_prime(n) else '':>6} {f(n):>8} {g(n):>10} {h(n):>13}")

for name, F in [("f", f), ("g", g), ("h", h)]:
    assert all((F(n) == 0) == is_prime(n) for n in range(2, N + 1)), name
print(f"\nFor 2 <= n <= {N}: f(n), g(n) and h(n) vanish exactly when n is prime.")

  n  prime     f(n)       g(n)          h(n)
  2    yes        0          0             0
  3    yes        0          0             0
  4              18        108          1080
  5    yes        0          0             0
  6             120       1260         24750
  7    yes        0          0             0
  8             270       4860        178200
  9             192       2592         58320
 10             504      14364        852390
 11    yes        0          0             0
 12            1680      51660       3644550
 13    yes        0          0             0
 14            1296      75816       9093060
 15            1536      43776       2424960
 16            2790     169020      24111000
 17    yes        0          0             0
 18            5160     367380      61592850
 19    yes        0          0             0
 20            6804     551124     117011790

For 2 <= n <= 1000: f(n), g(n) and h(n) vanish exactly when n is prime.


## 3. Some congruences

The values of $M_2$ and $M_3$ also satisfy Ramanujan-type congruences. The cell below checks

$$M_2(5n + 2) \equiv 0 \pmod 5, \qquad M_3(7n + 3) \equiv M_3(7n + 5) \equiv 0 \pmod 7$$

for every index up to $1000$.

In [4]:
def holds(a, t, b, m):
    # True if m divides M_a(tn + b) for every tn + b <= N
    return all(M[a][i] % m == 0 for i in range(b, N + 1, t))

for a, t, b, m in [(2, 5, 2, 5), (3, 7, 3, 7), (3, 7, 5, 7)]:
    assert holds(a, t, b, m)
    print(f"M_{a}({t}n + {b}) ≡ 0 (mod {m}) for {t}n + {b} <= {N}")

M_2(5n + 2) ≡ 0 (mod 5) for 5n + 2 <= 1000
M_3(7n + 3) ≡ 0 (mod 7) for 7n + 3 <= 1000
M_3(7n + 5) ≡ 0 (mod 7) for 7n + 5 <= 1000
